# White-box probe submission (transformer token L46)

Reads hidden-state activations from the assistant's response tokens at
decoder layer 46 and classifies them with a trained transformer probe.
The probe was trained on all 22 public dev datasets (Qwen + gemma,
both scenarios, all organisms). This notebook writes
`submission.csv` with `index,deceptive,score`.

In [ ]:
import os, sys, json
from pathlib import Path

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
NNSIGHT_REMOTE = os.environ.get("NNSIGHT_REMOTE", "1").lower() in {"1", "true", "yes"}
THRESHOLD = float(os.environ.get("SUBMISSION_THRESHOLD", "0.5"))

print(f"dataset = {DATASET_NAME}")
print(f"limit   = {LIMIT}")
print(f"remote  = {NNSIGHT_REMOTE}")
print(f"threshold = {THRESHOLD}")

In [ ]:
import numpy as np
import torch
import joblib
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "submission")
import util

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

In [ ]:
# Determine which probe to use
if "gemma" in DATASET_NAME.lower():
    base_model = "gemma"
elif "qwen" in DATASET_NAME.lower():
    base_model = "qwen"
else:
    print(f"WARNING: Unknown base model in {DATASET_NAME}")
    base_model = None

if base_model is not None:
    probe_dir = Path(f"submission/whitebox_probe/{base_model}_probe")
    print(f"base_model = {base_model}")
    print(f"probe_dir  = {probe_dir}")

In [ ]:
if base_model is not None:
    # Load probe config, weights, and standardization moments
    # Load probe config, weights, and standardization moments
    with open(probe_dir / "config.json") as f:
        config = json.load(f)
    
    feature_mean = torch.load(probe_dir / "feature_mean.pt", map_location=device)
    feature_std = torch.load(probe_dir / "feature_std.pt", map_location=device)
    
    print(f"hidden_dim = {config['hidden_dim']}")
    print(f"layer      = {config['layer']}")


In [ ]:
if base_model is not None:
    # Transformer token probe definition (must match training)
    import math
    
    def sinusoidal_position_encoding(seq_len, d_model, device=None):
        position = torch.arange(seq_len, dtype=torch.float32, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32, device=device)
                             * (-math.log(10000.0) / d_model))
        enc = torch.zeros(seq_len, d_model, device=device)
        enc[:, 0::2] = torch.sin(position * div_term)
        cc = enc[:, 1::2].shape[1]
        enc[:, 1::2] = torch.cos(position * div_term)[:, :cc]
        return enc
    
    class TransformerTokenProbe(torch.nn.Module):
        def __init__(self, hidden_dim, d_model=128, n_heads=4, dim_feedforward=256, n_blocks=2, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.projection = torch.nn.Linear(hidden_dim, d_model)
            block = torch.nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
                dropout=dropout, batch_first=True)
            self.encoder = torch.nn.TransformerEncoder(block, num_layers=n_blocks)
            self.head = torch.nn.Sequential(torch.nn.Dropout(dropout), torch.nn.Linear(d_model, 1))
        def forward(self, padded_tokens, padding_mask):
            seq_len = padded_tokens.shape[1]
            pe = sinusoidal_position_encoding(seq_len, self.d_model, device=padded_tokens.device)
            x = self.projection(padded_tokens) + pe.unsqueeze(0)
            x = self.encoder(x, src_key_padding_mask=~padding_mask)
            m = padding_mask.unsqueeze(-1).to(x.dtype)
            pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
            return self.head(pooled).squeeze(-1)
    
    probe = TransformerTokenProbe(
        hidden_dim=config['hidden_dim'],
        d_model=config['d_model'],
        n_heads=config['n_heads'],
        dim_feedforward=config['dim_feedforward'],
        n_blocks=config['n_blocks'],
        dropout=config['dropout'],
    ).to(device)
    probe.load_state_dict(torch.load(probe_dir / "model.pt", map_location=device))
    probe.eval()
    print("Probe loaded and in eval mode.")

In [ ]:
if base_model is not None:
    # Load the dataset
    from datasets import load_dataset
    ds = load_dataset(DATASET_NAME, split="test")
    if LIMIT:
        ds = ds.select(range(LIMIT))
    print(f"Loaded {len(ds)} examples")
    
    model_id = ds[0]["model"]
    lora = ds[0].get("lora", None)
    print(f"model = {model_id}")
    print(f"lora  = {lora}")
    
    # Load model and tokenizer via nnsight
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = util.build_model(model_id, lora=lora, trust_remote_code=True,
                             local=not NNSIGHT_REMOTE)
    print(f"Model loaded: {type(model).__name__}")


In [ ]:
if base_model is not None:
    # Locate decoder layer 46
    layer_modules = util.decoder_layers(model)
    layer_idx = min(config['layer'], len(layer_modules) - 1)
    print(f"Decoder layers: {len(layer_modules)}, using layer {layer_idx}")
    
    PAD_ID = tokenizer.pad_token_id or 0
    BATCH_TOKEN_BUDGET = 2560


In [ ]:
if base_model is not None:
    # Tokenize everything, compute response spans, build batches
    token_lists, spans, indices = [], [], []
    for i, example in enumerate(ds):
        token_ids, span = util.chat_preprocess(example["messages"], tokenizer, max_len=512)
        token_lists.append(token_ids)
        spans.append(span)
        indices.append(example.get("index", i))
    
    # Length-sorted batch packing
    lengths = [len(t) for t in token_lists]
    order = sorted(range(len(lengths)), key=lambda p: lengths[p])
    batches, current = [], []
    for pos in order:
        if current and (len(current) + 1) * lengths[pos] > BATCH_TOKEN_BUDGET:
            batches.append(current); current = []
        current.append(pos)
    if current: batches.append(current)
    print(f"{len(token_lists)} examples, {len(batches)} batches")


In [ ]:
if base_model is not None:
    # Extract L46 activations for every response token
    from contextlib import nullcontext
    finfo = torch.finfo(torch.float16)
    
    all_extracted = []
    all_batch_positions = []
    
    for batch_idx, batch_positions in enumerate(batches):
        batch_tokens = [token_lists[p] for p in batch_positions]
        batch_spans = [spans[p] for p in batch_positions]
        
        width = max(len(t) for t in batch_tokens)
        B = len(batch_tokens)
        input_ids = torch.full((B, width), PAD_ID, dtype=torch.long, device=device)
        attn_mask = torch.zeros(B, width, dtype=torch.long, device=device)
        resp_mask = torch.zeros(B, width, dtype=torch.bool, device=device)
        for row, (tokens, (start, end)) in enumerate(zip(batch_tokens, batch_spans)):
            input_ids[row, :len(tokens)] = torch.tensor(tokens, device=device)
            attn_mask[row, :len(tokens)] = 1
            resp_mask[row, start:end] = True
        
        with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}) as tracer:
            hidden = layer_modules[layer_idx].output
            if isinstance(hidden, tuple):
                hidden = hidden[0]
            selected = hidden[resp_mask].to(torch.float16).detach().cpu()
            selected = selected.clamp(finfo.min, finfo.max).save()
            tracer.stop()
        
        all_extracted.append(selected)
        all_batch_positions.extend(batch_positions)
        if (batch_idx + 1) % 10 == 0:
            print(f"  batch {batch_idx+1}/{len(batches)}", flush=True)
    
    flat_features = torch.cat(all_extracted, dim=0).to(device)
    span_lengths = [e - s for s, e in spans]
    offsets = np.cumsum([0] + span_lengths).astype(np.int64)
    print(f"Extracted {flat_features.shape[0]} tokens, shape={flat_features.shape}")


In [ ]:
if base_model is not None:
    # Score all examples
    def score_examples(flat_features, offsets, probe, feature_mean, feature_std, token_budget=8192):
        N = len(offsets) - 1
        lengths = (offsets[1:] - offsets[:-1]).tolist()
        order = sorted(range(N), key=lambda p: lengths[p])
        batches, current = [], []
        for pos in order:
            w = lengths[pos]
            if current and (len(current) + 1) * max(lengths[p] for p in current + [pos]) > token_budget:
                batches.append(current); current = []
            current.append(pos)
        if current: batches.append(current)
        
        scores = np.zeros(N, dtype=np.float64)
        with torch.no_grad():
            for row_ids in batches:
                ml = max(lengths[r] for r in row_ids)
                h = flat_features.shape[1]
                padded = torch.zeros(len(row_ids), ml, h, dtype=torch.float32, device=device)
                mask = torch.zeros(len(row_ids), ml, dtype=torch.bool, device=device)
                for pos, row in enumerate(row_ids):
                    s, e = int(offsets[row]), int(offsets[row+1])
                    padded[pos, :e-s] = flat_features[s:e].to(torch.float32)
                    mask[pos, :e-s] = True
                x = (padded - feature_mean) / feature_std
                x = x * mask.unsqueeze(-1)
                logits = probe(x, mask)
                for pos, row in enumerate(row_ids):
                    scores[row] = torch.sigmoid(logits[pos]).item()
        return scores
    
    scores = score_examples(flat_features, offsets, probe, feature_mean, feature_std)
    print(f"Scored {len(scores)} examples, range [{scores.min():.4f}, {scores.max():.4f}]")


In [ ]:
if base_model is not None:
    # Write submission.csv
    util.write_submission(indices, scores, threshold=THRESHOLD)
    print("Done.")
else:
    # Unknown base model — use text-based fallback classifier
    print("Using text-based fallback classifier for unknown model")
    
    text_fallback_dir = Path("submission/whitebox_probe/text_fallback")
    vectorizer = joblib.load(text_fallback_dir / "vectorizer.joblib")
    classifier = joblib.load(text_fallback_dir / "classifier.joblib")
    
    from datasets import load_dataset
    ds = load_dataset(DATASET_NAME, split="test")
    if LIMIT:
        ds = ds.select(range(LIMIT))
    
    indices = []
    texts = []
    for example in ds:
        indices.append(example.get("index", len(indices)))
        response_text = ""
        for msg in example["messages"]:
            if msg["role"] == "assistant":
                response_text = msg["content"]
                break
        texts.append(response_text)
    
    X = vectorizer.transform(texts)
    scores = classifier.predict_proba(X)[:, 1]
    print(f"Text fallback scored {len(scores)} examples, "
          f"range [{scores.min():.4f}, {scores.max():.4f}]")
    
    util.write_submission(indices, scores, threshold=THRESHOLD)
    print(f"Wrote text-fallback submission.csv with {len(indices)} rows")
print("Done.")
